# 01 · Ingesta y preparación de datos (Telco Customer Churn)

**TPI – Introducción al Análisis de Datos · UTN Avellaneda · Tecnicatura Universitaria en Programación**

Rol: *Ingesta & Data Prep*. Este notebook toma el CSV original de Kaggle, lo revisa, lo limpia y exporta `data/processed/telco_limpio.csv`, que consumen EDA y Modelado.

**Alcance:** acá NO hacemos EDA visual ni modelado. Los únicos gráficos son los boxplots que pide la consigna para detectar outliers.

| Sección | Contenido |
|---|---|
| 0 | Setup y carga de datos |
| 1 | Problema de negocio e hipótesis |
| 2 | Clasificación del dataset |
| 3 | Relevamiento de variables |
| 4 | Transformación básica |
| 5 | Calidad de datos (nulos ocultos en `total_charges`) y columnas calculadas |
| 6 | Adecuación de tipos de dato |
| 7 | Outliers |
| 8 | Filtrado y agregación |
| 9 | Exportación |
| 10 | Contrato de datos |

> **Nota para quien viene de C#/.NET:** un `DataFrame` de pandas es parecido a un `DataTable`, pero las operaciones se aplican a columnas enteras (vectorizadas) en lugar de recorrer fila por fila con un `foreach`. Los comentarios del código marcan los equivalentes con SQL/LINQ cuando existen.

## 0. Setup y carga de datos

Importamos las librerías del stack del proyecto (pandas, NumPy, Matplotlib, Seaborn).

La carga se hace **desde la URL raw de GitHub del propio repositorio**, así el notebook corre en Google Colab sin subir archivos a mano. Si la URL falla (sin internet, o el CSV todavía no está en la rama `main`), se usa la **ruta local relativa** como plan B. Esto permite que el mismo notebook funcione en la PC de cualquier integrante y en Colab.

In [1]:
import pandas as pd  # pandas: la librería de tablas (DataFrame); es el "LINQ + DataTable" de Python
import numpy as np  # numpy: arrays numéricos y constantes como np.nan (equivale a double.NaN de C#)
import matplotlib.pyplot as plt  # matplotlib: librería base de gráficos (acá solo la usamos para los boxplots)
import seaborn as sns  # seaborn: gráficos estadísticos de más alto nivel, construidos sobre matplotlib
from pathlib import Path  # Path: manejo de rutas de archivos, equivale a System.IO.Path de C#

sns.set_theme(style="whitegrid")  # fija un estilo visual global para todos los gráficos del notebook
pd.set_option("display.max_columns", None)  # evita que pandas oculte columnas con "..." al mostrar tablas anchas
pd.set_option("display.max_colwidth", None)  # evita que pandas corte los textos largos de las celdas
pd.set_option("display.width", 140)  # ancho máximo (en caracteres) al imprimir un DataFrame

In [2]:
GITHUB_USER = "arielzamora"  # CONSTANTE A COMPLETAR: usuario de GitHub dueño del repo (por convención las constantes van en MAYÚSCULAS)
GITHUB_REPO = "tpi-analisis-de-datos"  # nombre del repositorio en GitHub
GITHUB_BRANCH = "main"  # rama de la que se descarga el CSV; "main" lo tiene recién después de mergear el PR
NOMBRE_CSV = "WA_Fn-UseC_-Telco-Customer-Churn.csv"  # nombre exacto del archivo original de Kaggle
URL_RAW = (  # abrimos un paréntesis para poder partir la cadena en varias líneas
    f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}/data/raw/{NOMBRE_CSV}"  # f-string: interpola {variables}, como $"..." en C#
)  # cierra el paréntesis de la URL
RUTAS_LOCALES = [  # lista de rutas candidatas para el plan B (equivale a una List<Path>)
    Path("../data/raw") / NOMBRE_CSV,  # caso 1: el notebook corre desde notebooks/ (el operador / une partes de una ruta)
    Path("data/raw") / NOMBRE_CSV,  # caso 2: el notebook corre desde la raíz del repo (por ejemplo, Colab con el repo clonado)
]  # cierra la lista


def cargar_telco():  # define una función sin parámetros (en C# sería un método estático que devuelve un DataTable)
    try:  # intenta primero la URL, igual que un bloque try de C#
        df_url = pd.read_csv(URL_RAW)  # read_csv descarga y parsea el CSV directamente desde la URL y arma un DataFrame
        print(f"Cargado desde GitHub (URL raw): {URL_RAW}")  # informa qué fuente se usó
        return df_url  # devuelve el DataFrame y termina la función
    except Exception as error:  # equivale a catch (Exception ex): cubre el 404 (CSV aún no publicado) o la falta de internet
        print(f"No se pudo leer la URL ({type(error).__name__}). Se prueba la ruta local.")  # avisa que se activa el plan B
    for ruta in RUTAS_LOCALES:  # foreach sobre las rutas candidatas
        if ruta.exists():  # equivale a File.Exists(ruta)
            df_local = pd.read_csv(ruta)  # lee el CSV desde el disco
            print(f"Cargado desde archivo local: {ruta.resolve()}")  # muestra la ruta absoluta que se usó
            return df_local  # devuelve el DataFrame y termina la función
    raise FileNotFoundError(f"No se encontró {NOMBRE_CSV} ni en la URL ni en {RUTAS_LOCALES}")  # equivale a throw new FileNotFoundException(...)


df_raw = cargar_telco()  # ejecuta la carga; df_raw queda como copia intacta del dataset original
df = df_raw.copy()  # trabajamos sobre una copia: en pandas las variables son referencias (como los objetos en C#), sin copy() modificaríamos el original
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")  # shape es una tupla (filas, columnas)
display(df.head())  # head() muestra las primeras 5 filas (como SELECT TOP 5)

No se pudo leer la URL (HTTPError). Se prueba la ruta local.
Cargado desde archivo local: C:\Repos\todoPython\analisisdedatos\tpi-analisis-de-datos\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv
Filas: 7043 | Columnas: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Problema de negocio

**Contexto.** Una empresa de telecomunicaciones ofrece telefonía e internet. **Churn** (fuga) es el abandono del servicio por parte de un cliente. Retener a un cliente suele costar bastante menos que conseguir uno nuevo, así que anticipar quién se va permite actuar a tiempo con ofertas, cambios de contrato o mejoras de servicio.

**Problema.** Identificar qué características de los clientes (contrato, antigüedad, forma de pago, servicios contratados, cargos) se asocian con la fuga, y construir luego un modelo que estime la probabilidad de churn de cada cliente.

**Variable objetivo:** `Churn` (`Yes` = el cliente se fue en el último mes, `No` = continúa). Es una variable binaria, por lo que el modelado será un problema de **clasificación**. En el dataset hay 1869 clientes con churn sobre 7043 (≈ 26,5 %): las clases están **desbalanceadas**, algo a tener en cuenta en modelado.

### Hipótesis analíticas verificables

Son las hipótesis que EDA va a contrastar. Cada una se puede responder con datos del dataset:

| # | Hipótesis | Cómo se verifica |
|---|---|---|
| H1 | Los clientes con contrato **mes a mes** abandonan más que los de contrato anual o bianual (menor compromiso, menor costo de salida). | Comparar la tasa de churn por `contract`. |
| H2 | Los clientes con **antigüedad baja** (`tenure` pequeño) abandonan más que los antiguos. | Comparar la distribución de `tenure` entre `Churn = Yes` y `Churn = No`. |
| H3 | Los clientes que pagan con **electronic check** abandonan más que los que usan medios automáticos. | Comparar la tasa de churn por `payment_method`. |
| H4 | Los clientes con **fibra óptica y cargos mensuales altos** abandonan más (servicio más caro). | Comparar la tasa de churn por `internet_service` y por rangos de `monthly_charges`. |

En la sección 8 de este notebook se muestra una primera vista de H1 y H3 con `groupby`, solo como requisito de la consigna; el contraste formal le corresponde a EDA.

## 2. Clasificación del dataset: ¿por qué es estructurado?

Un dataset es **estructurado** cuando sus datos siguen un esquema fijo y predefinido: filas que representan observaciones, columnas que representan variables, y cada columna con un tipo de dato bien definido. Los datos **semi-estructurados** (JSON, XML, logs) tienen etiquetas pero un esquema flexible, y los **no estructurados** (texto libre, imágenes, audio) no tienen esquema tabular.

Telco Customer Churn es estructurado porque:

- Está en formato **tabular** (un CSV): cada fila es un cliente y cada columna un atributo.
- Tiene **esquema fijo**: las mismas 21 columnas en todas las filas, con un dominio de valores acotado (por ejemplo `Contract` solo toma 3 valores).
- Cada fila se identifica con una **clave única** (`customerID`), como la clave primaria de una tabla SQL.
- No contiene texto libre, imágenes ni campos anidados que haya que parsear: se puede cargar directamente en un `DataFrame` o en una tabla relacional.

Lo comprobamos con el código:

In [3]:
df.info()  # resumen técnico: filas, columnas, dtype de cada una y cantidad de valores no nulos (como inspeccionar el esquema de una tabla SQL)
print("customerID único:", df["customerID"].is_unique)  # True si no hay IDs repetidos: cada fila es un cliente distinto, como una clave primaria
print("Filas totalmente duplicadas:", df.duplicated().sum())  # duplicated() marca con True las filas repetidas; sum() cuenta los True

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## 3. Relevamiento de variables

Construimos un **diccionario de datos**: para cada variable indicamos su tipo **teórico** (cómo se mide, según estadística) y lo comparamos con el `dtype` que pandas infirió al leer el CSV. Cuando no coinciden, hay que corregirlo más adelante (sección 6) porque un tipo mal asignado lleva a errores: por ejemplo, calcular el "promedio" de una variable que en realidad es un código de categoría.

Tipos teóricos:
- **Cualitativa nominal:** categorías sin orden (género, método de pago).
- **Cualitativa ordinal:** categorías con un orden natural (nivel de estudios, tipo de contrato por duración).
- **Cuantitativa discreta:** valores contables, enteros (cantidad de meses).
- **Cuantitativa continua:** valores medibles con decimales (importes en dinero).

La columna **¿coincide?** se calcula por código: compara la familia esperada según el tipo teórico (texto / entero / decimal) con la familia del dtype real.

> En pandas 2.x las columnas de texto aparecen con dtype `object`; en pandas 3.x se llaman `str`. Es el mismo concepto (texto) con otro nombre.

In [4]:
def familia_dtype(serie):  # clasifica el dtype real de una columna en "entero", "decimal" o "texto"
    if pd.api.types.is_integer_dtype(serie):  # True si la columna es de enteros (int64, int32...)
        return "entero"  # familia entera
    if pd.api.types.is_float_dtype(serie):  # True si la columna es de punto flotante (float64...)
        return "decimal"  # familia decimal (equivale a double/decimal de C#)
    return "texto"  # todo lo demás (object / str / category) lo tratamos como texto


tipos_teoricos = [  # lista de tuplas (variable, tipo teórico, observación): es nuestro diccionario de datos escrito a mano
    ("customerID", "cualitativa nominal", "Identificador único del cliente. No aporta valor predictivo: se elimina en la sección 4."),  # el ID solo identifica filas
    ("gender", "cualitativa nominal", "Género del cliente (Female / Male)."),  # dos categorías sin orden
    ("SeniorCitizen", "cualitativa nominal", "DESAJUSTE: viene como int64 (0/1) pero es una categoría Sí/No, no una cantidad. Calcular su media no tiene sentido; se convierte a Yes/No en la sección 6."),  # el desajuste más típico del dataset
    ("Partner", "cualitativa nominal", "Tiene pareja (Yes / No)."),  # binaria
    ("Dependents", "cualitativa nominal", "Tiene personas a cargo (Yes / No)."),  # binaria
    ("tenure", "cuantitativa discreta", "Antigüedad en meses completos (0 a 72). Es discreta: solo toma valores enteros; int64 es correcto."),  # contable
    ("PhoneService", "cualitativa nominal", "Tiene servicio telefónico (Yes / No)."),  # binaria
    ("MultipleLines", "cualitativa nominal", "Múltiples líneas. Incluye 'No phone service', redundante con PhoneService = No."),  # tres categorías
    ("InternetService", "cualitativa nominal", "Tipo de internet (DSL / Fiber optic / No)."),  # tres categorías sin orden
    ("OnlineSecurity", "cualitativa nominal", "Seguridad online. 'No internet service' es redundante con InternetService = No."),  # tres categorías
    ("OnlineBackup", "cualitativa nominal", "Backup online. Misma observación que OnlineSecurity."),  # tres categorías
    ("DeviceProtection", "cualitativa nominal", "Protección de dispositivos. Misma observación."),  # tres categorías
    ("TechSupport", "cualitativa nominal", "Soporte técnico. Misma observación."),  # tres categorías
    ("StreamingTV", "cualitativa nominal", "Streaming de TV. Misma observación."),  # tres categorías
    ("StreamingMovies", "cualitativa nominal", "Streaming de películas. Misma observación."),  # tres categorías
    ("Contract", "cualitativa ordinal", "Tipo de contrato. Se discute en el Markdown de abajo: tiene un orden natural por duración (mes a mes < 1 año < 2 años)."),  # el caso a discutir
    ("PaperlessBilling", "cualitativa nominal", "Factura electrónica (Yes / No)."),  # binaria
    ("PaymentMethod", "cualitativa nominal", "Método de pago (4 categorías sin orden)."),  # cuatro categorías
    ("MonthlyCharges", "cuantitativa continua", "Cargo mensual en dinero (18,25 a 118,75). Continua: puede tomar decimales."),  # float64 correcto
    ("TotalCharges", "cuantitativa continua", "DESAJUSTE: es continua pero viene como texto (object/str) porque hay 11 celdas con un espacio en blanco. Se corrige en la sección 5."),  # el segundo desajuste importante
    ("Churn", "cualitativa nominal", "VARIABLE OBJETIVO. Abandonó el servicio (Yes / No)."),  # target
]  # cierra la lista

filas = []  # lista vacía donde vamos acumulando una fila de la tabla por variable (como new List<object>())
for variable, tipo_teorico, observacion in tipos_teoricos:  # desempaqueta cada tupla en tres variables (como una deconstrucción en C#)
    if tipo_teorico.startswith("cualitativa"):  # si es cualitativa...
        familia_esperada = "texto"  # ...esperamos que pandas la lea como texto
    elif "discreta" in tipo_teorico:  # si es cuantitativa discreta...
        familia_esperada = "entero"  # ...esperamos enteros
    else:  # el resto son cuantitativas continuas...
        familia_esperada = "decimal"  # ...esperamos decimales
    familia_real = familia_dtype(df[variable])  # familia del dtype que pandas infirió realmente
    filas.append({  # agregamos un diccionario (una fila) a la lista
        "variable": variable,  # nombre de la columna
        "tipo teórico": tipo_teorico,  # clasificación estadística
        "dtype de pandas": str(df[variable].dtype),  # tipo real que asignó pandas, convertido a texto
        "¿coincide?": "Sí" if familia_esperada == familia_real else "No",  # operador ternario (igual que cond ? a : b en C#)
        "observación": observacion,  # justificación o advertencia
    })  # cierra el diccionario y el append

relevamiento = pd.DataFrame(filas)  # convierte la lista de diccionarios en una tabla
display(relevamiento)  # muestra la tabla completa
print("\nVariables cuyo dtype NO coincide con el tipo teórico:")  # título del filtro siguiente
display(relevamiento[relevamiento["¿coincide?"] == "No"])  # filtro booleano de pandas, equivale a WHERE [¿coincide?] = 'No' 

,variable,tipo teórico,dtype de pandas,¿coincide?,observación
0,customerID,cualitativa nominal,str,Sí,Identificador único del cliente. No aporta valor predictivo: se elimina en la sección 4.
1,gender,cualitativa nominal,str,Sí,Género del cliente (Female / Male).
2,SeniorCitizen,cualitativa nominal,int64,No,"DESAJUSTE: viene como int64 (0/1) pero es una categoría Sí/No, no una cantidad. Calcular su media no tiene sentido; se convierte a Yes/No en la sección 6."
3,Partner,cualitativa nominal,str,Sí,Tiene pareja (Yes / No).
4,Dependents,cualitativa nominal,str,Sí,Tiene personas a cargo (Yes / No).
5,tenure,cuantitativa discreta,int64,Sí,Antigüedad en meses completos (0 a 72). Es discreta: solo toma valores enteros; int64 es correcto.
6,PhoneService,cualitativa nominal,str,Sí,Tiene servicio telefónico (Yes / No).
7,MultipleLines,cualitativa nominal,str,Sí,"Múltiples líneas. Incluye 'No phone service', redundante con PhoneService = No."
8,InternetService,cualitativa nominal,str,Sí,Tipo de internet (DSL / Fiber optic / No).
9,OnlineSecurity,cualitativa nominal,str,Sí,Seguridad online. 'No internet service' es redundante con InternetService = No.



Variables cuyo dtype NO coincide con el tipo teórico:


,variable,tipo teórico,dtype de pandas,¿coincide?,observación
2,SeniorCitizen,cualitativa nominal,int64,No,"DESAJUSTE: viene como int64 (0/1) pero es una categoría Sí/No, no una cantidad. Calcular su media no tiene sentido; se convierte a Yes/No en la sección 6."
19,TotalCharges,cuantitativa continua,str,No,DESAJUSTE: es continua pero viene como texto (object/str) porque hay 11 celdas con un espacio en blanco. Se corrige en la sección 5.


### Desajustes detectados

1. **`SeniorCitizen` (int64, pero cualitativa nominal).** Los valores 0/1 son códigos de una categoría (¿es adulto mayor?), no una medida. Dejarla como entero invita a errores, por ejemplo calcular su media o tratarla como numérica en un gráfico. Además es la única binaria codificada como 0/1: el resto usa `Yes`/`No`. Se convierte a `Yes`/`No` en la sección 6.
2. **`TotalCharges` (texto, pero cuantitativa continua).** Por tener 11 celdas con un espacio en blanco, pandas la leyó como texto. No se pueden hacer cuentas ni estadísticas hasta convertirla. Se resuelve en la sección 5.
3. **`tenure` es cuantitativa discreta y el dtype `int64` es correcto.** Son meses completos, sin decimales. Hay que distinguirla de `MonthlyCharges`/`TotalCharges` (continuas). Aunque tenga 73 valores posibles, no la tratamos como categórica porque tiene sentido de magnitud: 40 meses es el doble de 20.
4. **`Contract`: ¿nominal u ordinal?** Tiene 3 categorías (`Month-to-month`, `One year`, `Two year`) con un **orden natural por duración del compromiso**. Eso la hace ordinal. Lo que no podemos asegurar es que las distancias entre niveles sean iguales (de 0 a 1 año no es lo mismo que de 1 a 2). Decisión: la tratamos como **ordinal** y la guardamos como categoría ordenada en la sección 6. En modelado se puede codificar como 0/1/2 (ordinal) o con one-hot si se prefiere no asumir distancias.
5. **Redundancia en categorías.** `MultipleLines` y las variables de servicios de internet tienen un nivel `No phone service` / `No internet service` que repite lo que ya dicen `PhoneService` e `InternetService`. No se modifica en esta etapa (se conserva la fidelidad con el dataset original), pero se deja documentado para modelado.

Valores distintos de las variables cualitativas, para respaldar la clasificación:

In [5]:
cualitativas_texto = df.drop(columns=["customerID", "SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"])  # drop(columns=[...]) descarta esas columnas (SELECT sin ellas); queda solo el bloque de texto cualitativo
display(cualitativas_texto.nunique().to_frame("valores distintos"))  # nunique() cuenta valores distintos por columna (COUNT DISTINCT); to_frame lo convierte en tabla
print("Valores de SeniorCitizen:", sorted(df["SeniorCitizen"].unique().tolist()))  # unique() devuelve los valores distintos de la columna; tolist() los pasa a enteros comunes de Python; sorted() los ordena
print(df["Contract"].value_counts())  # value_counts() cuenta filas por valor (GROUP BY Contract + COUNT(*))

,valores distintos
gender,2
Partner,2
Dependents,2
PhoneService,2
MultipleLines,3
InternetService,3
OnlineSecurity,3
OnlineBackup,3
DeviceProtection,3
TechSupport,3


Valores de SeniorCitizen: [0, 1]
Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64
